In [9]:
import os
import pandas as pd
import glob
from datetime import datetime
import os
import re
import numpy as np

In [ ]:
DATA_DIR   = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
EXTRACT_DIR = os.path.join(DATA_DIR, "csv")

print("Looking for files in:", EXTRACT_DIR)

csv_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "JC-*.csv")))
print(f"Found {len(csv_files)} files")
print("First few:", [os.path.basename(f) for f in csv_files[:5]])
print("Last few: ", [os.path.basename(f) for f in csv_files[-5:]])

if not csv_files:
    raise FileNotFoundError("No JC CSV files found. Check extraction step.")

Looking for files in: c:\Users\aghab\Desktop\coding\citibike\data\csv
Found 14 files
First few: ['JC-202412-citibike-tripdata.csv', 'JC-202501-citibike-tripdata.csv', 'JC-202502-citibike-tripdata.csv', 'JC-202503-citibike-tripdata.csv', 'JC-202504-citibike-tripdata.csv']
Last few:  ['JC-202509-citibike-tripdata.csv', 'JC-202510-citibike-tripdata.csv', 'JC-202511-citibike-tripdata.csv', 'JC-202512-citibike-tripdata.csv', 'JC-202601-citibike-tripdata.csv']


In [5]:
sample_file = csv_files[0]
print(f"\nInspecting sample file: {os.path.basename(sample_file)}")

df_sample = pd.read_csv(sample_file, nrows=10000, low_memory=False)
print("\nShape:", df_sample.shape)
print("\nColumns:")
print(df_sample.columns.tolist())
print("\nData types:")
print(df_sample.dtypes)
print("\nFirst 3 rows:")
display(df_sample.head(3))


Inspecting sample file: JC-202412-citibike-tripdata.csv

Shape: (10000, 13)

Columns:
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

Data types:
ride_id                   str
rideable_type             str
started_at                str
ended_at                  str
start_station_name        str
start_station_id          str
end_station_name          str
end_station_id            str
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual             str
dtype: object

First 3 rows:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,28A17ACD224CD80B,electric_bike,2024-12-06 17:50:49.428,2024-12-06 17:54:20.070,Oakland Ave,JC022,Hilltop,JC019,40.737604,-74.052478,40.731169,-74.057574,member
1,3508393A86FBD357,classic_bike,2024-12-14 11:01:00.309,2024-12-14 11:12:01.382,Oakland Ave,JC022,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.737604,-74.052478,40.735938,-74.030305,member
2,75FA4C03A1447401,electric_bike,2024-12-24 08:07:17.475,2024-12-24 08:14:14.612,Oakland Ave,JC022,Leonard Gordon Park,JC080,40.737604,-74.052478,40.745910,-74.057271,member


In [6]:
# Check column consistency across all files
print("\nChecking column consistency across files...")

all_columns = set()
column_variations = {}

for f in csv_files:
    df_temp = pd.read_csv(f, nrows=1)
    cols = df_temp.columns.tolist()
    all_columns.update(cols)
    
    key = tuple(sorted(cols))
    if key not in column_variations:
        column_variations[key] = []
    column_variations[key].append(os.path.basename(f))

print(f"Unique column sets found: {len(column_variations)}")

if len(column_variations) > 1:
    print("WARNING: Different column sets detected!")
    for cols_tuple, files in column_variations.items():
        print(f"  → {len(files)} files with columns:")
        print("   ", list(cols_tuple))
        print("   Files:", files)
else:
    print("All files have the same columns ✓")


Checking column consistency across files...
Unique column sets found: 1
All files have the same columns ✓


In [7]:
# Load & concatenate all files
print("\nLoading and concatenating all files... (this may take 1–3 minutes)")

dfs = []
for filepath in csv_files:
    filename = os.path.basename(filepath)
    print(f"  Loading {filename} ...", end="")
    
    df = pd.read_csv(
        filepath,
        parse_dates=['started_at', 'ended_at'],   # try to parse dates early
        date_format='%Y-%m-%d %H:%M:%S',          # common Citi Bike format
        low_memory=False
    )
    
    # Add source month for traceability
    month_str = filename[3:9]  # JC-YYYYMM-...
    df['source_month'] = month_str
    
    dfs.append(df)
    print(f" {df.shape[0]:,} rows")

# Concatenate
all_trips = pd.concat(dfs, ignore_index=True, sort=False)
print(f"\nFinal combined shape: {all_trips.shape}")
print(f"Memory usage: {all_trips.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


Loading and concatenating all files... (this may take 1–3 minutes)
  Loading JC-202412-citibike-tripdata.csv ... 54,833 rows
  Loading JC-202501-citibike-tripdata.csv ... 50,611 rows
  Loading JC-202502-citibike-tripdata.csv ... 45,255 rows
  Loading JC-202503-citibike-tripdata.csv ... 73,293 rows
  Loading JC-202504-citibike-tripdata.csv ... 81,553 rows
  Loading JC-202505-citibike-tripdata.csv ... 93,227 rows
  Loading JC-202506-citibike-tripdata.csv ... 97,124 rows
  Loading JC-202507-citibike-tripdata.csv ... 107,726 rows
  Loading JC-202508-citibike-tripdata.csv ... 108,441 rows
  Loading JC-202509-citibike-tripdata.csv ... 116,071 rows
  Loading JC-202510-citibike-tripdata.csv ... 104,205 rows
  Loading JC-202511-citibike-tripdata.csv ... 76,724 rows
  Loading JC-202512-citibike-tripdata.csv ... 48,474 rows
  Loading JC-202601-citibike-tripdata.csv ... 40,044 rows

Final combined shape: (1097581, 14)
Memory usage: 770.6 MB


In [8]:
# Basic quality checks
print("\nMissing values (%):")
missing = all_trips.isna().mean().sort_values(ascending=False) * 100
print(missing[missing > 0].round(2))

print("\nDate range:")
print("Started at:", all_trips['started_at'].min(), "–", all_trips['started_at'].max())
print("Ended at:  ", all_trips['ended_at'].min(),   "–", all_trips['ended_at'].max())

print("\nUnique start stations:", all_trips['start_station_name'].nunique())
print("Unique end stations:  ", all_trips['end_station_name'].nunique())

print("\nRideable types:", all_trips['rideable_type'].value_counts(dropna=False))
print("\nMember vs casual:", all_trips['member_casual'].value_counts(dropna=False))


Missing values (%):
end_station_id        0.49
end_lat               0.39
end_lng               0.39
end_station_name      0.35
start_station_name    0.00
start_station_id      0.00
start_lng             0.00
start_lat             0.00
dtype: float64

Date range:
Started at: 2024-11-30 23:32:54.155 – 2026-01-31 23:47:07.645
Ended at:   2024-12-01 00:01:10.086 – 2026-01-31 23:53:17.737

Unique start stations: 112
Unique end stations:   499

Rideable types: rideable_type
electric_bike    709627
classic_bike     387954
Name: count, dtype: int64

Member vs casual: member_casual
member    860145
casual    237436
Name: count, dtype: int64


In [10]:
# ─── Save full dataset as CSV ─────────────────────────────────────
output_csv_path = os.path.join(DATA_DIR, "jc_trips_combined_202412_202601.csv")

print(f"Saving full dataset ({all_trips.shape[0]:,} rows, {all_trips.shape[1]} columns)...")
all_trips.to_csv(output_csv_path, index=False)
print(f"Done! File saved to:")
print(f"  {output_csv_path}")
print(f"File size: {os.path.getsize(output_csv_path) / (1024**2):.1f} MB")

Saving full dataset (1,097,581 rows, 14 columns)...
Done! File saved to:
  c:\Users\aghab\Desktop\coding\citibike\data\jc_trips_combined_202412_202601.csv
File size: 215.8 MB


In [3]:
# ── Load ──────────────────────────────────────────────────────────
DATA_DIR  = "../data"
RAW_CSV   = os.path.join(DATA_DIR, "jc_trips_combined_202412_202601.csv")

print("Loading CSV...")
raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"  Shape: {raw.shape[0]:,} rows × {raw.shape[1]} cols")
print(f"  Memory: {raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Loading CSV...
  Shape: 1,097,581 rows × 14 cols
  Memory: 713.0 MB


In [4]:
def snapshot(df, label=""):
    """Quick shape + missing-value summary."""
    print(f"\n{'─'*55}")
    if label:
        print(f"  {label}")
    print(f"  Rows: {len(df):,}   Cols: {df.shape[1]}")
    missing = df.isna().mean().mul(100)
    missing = missing[missing > 0].sort_values(ascending=False).round(2)
    if len(missing):
        print("  Missing (%):")
        for col, pct in missing.items():
            print(f"    {col:<30} {pct:.2f}%")
    else:
        print("  No missing values ✓")
    print(f"{'─'*55}")

snapshot(raw, "RAW")


───────────────────────────────────────────────────────
  RAW
  Rows: 1,097,581   Cols: 14
  Missing (%):
    end_station_id                 0.49%
    end_lng                        0.39%
    end_lat                        0.39%
    end_station_name               0.35%
    start_station_name             0.00%
    start_station_id               0.00%
    start_lng                      0.00%
    start_lat                      0.00%
───────────────────────────────────────────────────────


In [5]:
df = raw.copy()

# ── Datetimes (microseconds present, so no format string) ─────────
for col in ['started_at', 'ended_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

nat_count = df[['started_at', 'ended_at']].isna().sum()
print("NaT counts after parsing:")
print(nat_count.to_string())

# ── Categories (low-cardinality strings) ─────────────────────────
for col in ['rideable_type', 'member_casual']:
    df[col] = df[col].astype('category')

# ── Downcasted floats (float64 → float32 saves ~50% memory) ──────
for col in ['start_lat', 'start_lng', 'end_lat', 'end_lng']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

# ── String columns ────────────────────────────────────────────────
for col in ['ride_id', 'start_station_id', 'start_station_id',
            'end_station_name',  'end_station_id',
            'start_station_name']:
    df[col] = df[col].astype('string')

print(f"\nMemory after dtype fix: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
snapshot(df, "After dtype fix")

NaT counts after parsing:
started_at    0
ended_at      0

Memory after dtype fix: 409.1 MB

───────────────────────────────────────────────────────
  After dtype fix
  Rows: 1,097,581   Cols: 14
  Missing (%):
    end_station_id                 0.49%
    end_lng                        0.39%
    end_lat                        0.39%
    end_station_name               0.35%
    start_station_name             0.00%
    start_station_id               0.00%
    start_lng                      0.00%
    start_lat                      0.00%
───────────────────────────────────────────────────────


In [6]:
n_before = len(df)

# Exact duplicate rows
df = df.drop_duplicates()
print(f"Duplicate rows removed:    {n_before - len(df):,}")

# Duplicate ride_ids (keep first occurrence)
n_before = len(df)
df = df.drop_duplicates(subset='ride_id', keep='first')
print(f"Duplicate ride_ids removed: {n_before - len(df):,}")
print(f"Remaining: {len(df):,}")

Duplicate rows removed:    0
Duplicate ride_ids removed: 0
Remaining: 1,097,581


In [7]:
df['duration_sec'] = (df['ended_at'] - df['started_at']).dt.total_seconds()
df['duration_min'] = df['duration_sec'] / 60

print("Duration distribution (min) — before filtering:")
print(df['duration_min'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).round(2))

# ── Remove physically impossible trips ───────────────────────────
# < 1 min  → docking error / accidental unlock
# > 24 hrs → almost certainly a forgotten return / data error
MIN_DURATION_SEC =  60       # 1 minute
MAX_DURATION_SEC = 86_400    # 24 hours

mask_valid = df['duration_sec'].between(MIN_DURATION_SEC, MAX_DURATION_SEC)
n_dropped  = (~mask_valid).sum()
print(f"\nTrips outside [{MIN_DURATION_SEC}s, {MAX_DURATION_SEC}s]: {n_dropped:,} "
      f"({n_dropped / len(df) * 100:.2f}%)")

df = df[mask_valid].copy()
print(f"Remaining: {len(df):,}")

print("\nDuration distribution (min) — after filtering:")
print(df['duration_min'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).round(2))

Duration distribution (min) — before filtering:
count    1097581.00
mean          10.16
std           41.88
min          -57.02
1%             1.39
5%             2.14
25%            3.94
50%            5.98
75%            9.45
95%           23.95
99%           60.27
max         5656.05
Name: duration_min, dtype: float64

Trips outside [60s, 86400s]: 482 (0.04%)
Remaining: 1,097,099

Duration distribution (min) — after filtering:
count    1097099.00
mean           9.51
std           27.71
min            1.00
1%             1.39
5%             2.14
25%            3.94
50%            5.98
75%            9.44
95%           23.82
99%           59.05
max         1439.31
Name: duration_min, dtype: float64


In [10]:
# Jersey City bounding box (generous, covers all realistic JC/Hoboken stations)
LAT_MIN, LAT_MAX =  40.60,  40.85
LNG_MIN, LNG_MAX = -74.25, -73.90

def flag_bad_coords(df, lat_col, lng_col, label):
    bad_lat = ~df[lat_col].between(LAT_MIN, LAT_MAX) & df[lat_col].notna()
    bad_lng = ~df[lng_col].between(LNG_MIN, LNG_MAX) & df[lng_col].notna()
    bad     = bad_lat | bad_lng
    print(f"  {label}: {bad.sum():,} out-of-range coordinates")
    return bad

bad_start = flag_bad_coords(df, 'start_lat', 'start_lng', 'start')
bad_end   = flag_bad_coords(df, 'end_lat',   'end_lng',   'end')

# Null out bad coords (don't drop — trip duration is still valid)
df.loc[bad_start, ['start_lat', 'start_lng']] = np.nan
df.loc[bad_end,   ['end_lat',   'end_lng']]   = np.nan

snapshot(df, "After coordinate validation")

  start: 0 out-of-range coordinates
  end: 7 out-of-range coordinates

───────────────────────────────────────────────────────
  After coordinate validation
  Rows: 1,097,099   Cols: 16
  Missing (%):
    end_station_id                 0.45%
    end_lng                        0.35%
    end_lat                        0.35%
    end_station_name               0.31%
    start_station_name             0.00%
    start_station_id               0.00%
    start_lng                      0.00%
    start_lat                      0.00%
───────────────────────────────────────────────────────


In [11]:
# ── Trim whitespace & normalise case ─────────────────────────────
for col in ['start_station_name', 'end_station_name']:
    df[col] = (df[col]
               .str.strip()
               .str.replace(r'\s+', ' ', regex=True))   # collapse internal spaces

# ── Audit: stations that appear with multiple IDs ─────────────────
# (data quality signal — not fixed here, flagged for awareness)
name_to_ids = (df.groupby('start_station_name')['start_station_id']
                 .nunique()
                 .sort_values(ascending=False))

multi_id = name_to_ids[name_to_ids > 1]
if len(multi_id):
    print(f"Station names with >1 station_id: {len(multi_id)}")
    print(multi_id.head(10).to_string())
else:
    print("All station names map to a single ID ✓")

# ── Station network flag (JC vs Hoboken vs NYC spillover) ─────────
def classify_station(station_id):
    if pd.isna(station_id):
        return pd.NA
    sid = str(station_id)
    if sid.startswith('JC'):
        return 'jersey_city'
    elif sid.startswith('HB'):
        return 'hoboken'
    else:
        return 'other'          # NYC or unknown

df['start_network'] = df['start_station_id'].apply(classify_station).astype('category')
df['end_network']   = df['end_station_id'].apply(classify_station).astype('category')

print("\nStart network distribution:")
print(df['start_network'].value_counts(dropna=False).to_string())
print("\nEnd network distribution:")
print(df['end_network'].value_counts(dropna=False).to_string())

All station names map to a single ID ✓

Start network distribution:
start_network
jersey_city    598314
hoboken        498781
NaN                 4

End network distribution:
end_network
jersey_city    590888
hoboken        497704
NaN              4936
other            3571


In [12]:
# Ensure only expected values exist — catches typos / schema drift
VALID_RIDEABLE = {'electric_bike', 'classic_bike', 'docked_bike'}
VALID_MEMBER   = {'member', 'casual'}

bad_rideable = ~df['rideable_type'].isin(VALID_RIDEABLE)
bad_member   = ~df['member_casual'].isin(VALID_MEMBER)

print(f"Unexpected rideable_type values: {bad_rideable.sum():,}")
if bad_rideable.any():
    print(df.loc[bad_rideable, 'rideable_type'].value_counts().to_string())

print(f"Unexpected member_casual values:  {bad_member.sum():,}")
if bad_member.any():
    print(df.loc[bad_member, 'member_casual'].value_counts().to_string())

Unexpected rideable_type values: 0
Unexpected member_casual values:  0


In [13]:
# These columns are non-negotiable for any downstream analysis
CRITICAL_COLS = [
    'ride_id', 'started_at', 'ended_at',
    'start_station_name', 'end_station_name',
    'member_casual', 'rideable_type',
]

n_before = len(df)
df = df.dropna(subset=CRITICAL_COLS)
print(f"Dropped {n_before - len(df):,} rows missing critical fields")
print(f"Remaining: {len(df):,}")

snapshot(df, "After dropping critical nulls")

Dropped 3,414 rows missing critical fields
Remaining: 1,093,685

───────────────────────────────────────────────────────
  After dropping critical nulls
  Rows: 1,093,685   Cols: 18
  Missing (%):
    end_station_id                 0.14%
    end_network                    0.14%
    end_lat                        0.12%
    end_lng                        0.12%
───────────────────────────────────────────────────────


In [15]:
# ── Fix source_month dtype ────────────────────────────────────────
df['source_month'] = df['source_month'].astype(str).astype('category')

# ── Cleaning report ───────────────────────────────────────────────
n_raw   = len(raw)
n_clean = len(df)
n_lost  = n_raw - n_clean

print("=" * 55)
print("  CLEANING SUMMARY")
print("=" * 55)
print(f"  Raw trips:     {n_raw:>10,}")
print(f"  Clean trips:   {n_clean:>10,}  ({n_clean/n_raw*100:.2f}%)")
print(f"  Removed:       {n_lost:>10,}  ({n_lost/n_raw*100:.2f}%)")
print(f"  Date range:    {df['started_at'].min().date()} → {df['started_at'].max().date()}")
print(f"  Columns:       {df.shape[1]}")
print(f"  Memory:        {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("=" * 55)
print("\nFinal dtypes:")
print(df.dtypes.to_string())

# ── Save ──────────────────────────────────────────────────────────
OUTPUT_PATH = os.path.join(DATA_DIR, "jc_trips_clean.csv")
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved → {OUTPUT_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024**2:.1f} MB")

  CLEANING SUMMARY
  Raw trips:      1,097,581
  Clean trips:    1,093,685  (99.65%)
  Removed:            3,896  (0.35%)
  Date range:    2024-11-30 → 2026-01-31
  Columns:       18
  Memory:        427.5 MB

Final dtypes:
ride_id                       string
rideable_type               category
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name            string
start_station_id              string
end_station_name              string
end_station_id                string
start_lat                    float32
start_lng                    float32
end_lat                      float32
end_lng                      float32
member_casual               category
source_month                category
duration_sec                 float64
duration_min                 float64
start_network               category
end_network                 category

Saved → ../data\jc_trips_clean.csv
File size: 238.6 MB


## Data Cleaning — Completed

**Input:** `jc_trips_combined_202412_202601.csv` — 1,097,581 raw trips (Nov 2024 – Jan 2026)  
**Output:** `jc_trips_clean.csv` — 1,093,685 trips (99.65% retained)

### What Was Done

**Dtype optimisation**
- Parsed `started_at` / `ended_at` as `datetime64` with microsecond support (`errors='coerce'`)
- Cast `rideable_type`, `member_casual`, `source_month` to `category`
- Downcast coordinates to `float32`
- Zero NaT values after parsing

**Deduplication**
- 0 duplicate rows
- 0 duplicate `ride_id` values

**Duration filtering**
- Calculated `duration_sec` and `duration_min` from start/end timestamps
- Removed trips outside the valid range [60s, 86400s]
- 482 trips removed (0.04%) — accidental unlocks and forgotten returns

**Coordinate validation**
- Validated against Jersey City / Hoboken bounding box (lat 40.60–40.85, lng -74.25–-73.90)
- 0 bad start coordinates, 7 bad end coordinates → nulled (rows retained)

**Station standardisation**
- Stripped and collapsed whitespace on all station name fields
- Confirmed all station names map to a single `station_id` — no conflicts
- Added `start_network` / `end_network` flags: `jersey_city`, `hoboken`, `other`

**Categorical audit**
- 0 unexpected values in `rideable_type` or `member_casual`

**Critical field filtering**
- Dropped 3,414 rows missing one or more of: `ride_id`, `started_at`, `ended_at`, `start_station_name`, `end_station_name`, `member_casual`, `rideable_type`

### Final Dataset

| | |
|---|---|
| Trips | 1,093,685 |
| Date range | 2024-11-30 → 2026-01-31 |
| Columns | 18 |
| Memory | 427.5 MB |
| File size | 238.6 MB |
| Missing values | end fields only (0.12–0.14%) — dockless electric bike trips |

### Remaining Missing Values (Expected)

| Column | Missing |
|---|---|
| `end_station_id` | 0.14% |
| `end_network` | 0.14% |
| `end_lat` / `end_lng` | 0.12% |

These are dockless electric bike trips that ended without a station dock — structurally missing and not imputable. Handled per-analysis as needed.

### Next Step
Feature engineering: Haversine trip distance, temporal flags (hour, weekday, season, rush hour), station net flow.